# Effects of Preprocessing Schemes on Classifier Performance under Varying Percentages of Missing Values


**Authors:** Christian R. Carpio and Luis E. Benavides

## 1. Imports & Warnings Configuration

In [ ]:
import os
import re
import warnings
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass

import numpy as np
import pandas as pd
from numpy.random import default_rng
from pandas.api.types import is_numeric_dtype

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    MinMaxScaler, OneHotEncoder, OrdinalEncoder, 
    RobustScaler, StandardScaler
)
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

## 2. Experiment Configuration

In [ ]:
@dataclass
class ExperimentConfig:
    seed: int = 2025
    target_column: str = "HeartDisease"
    mcar_rates: tuple = (0.10, 0.20, 0.30)
    cv_folds: int = 5

    results_dir: str = "results"
    preprocessed_dir: str = "preprocessed_datasets"
    missing_dir: str = "missing_values_datasets"
    anova_dir: str = "results/anova"
    tables_dir: str = "results/tables"
    rankings_dir: str = "results/tables/rankings"
    graphs_dir: str = "results/graphs"

    knn_neighbors: int = 5
    mice_max_iter: int = 25
    mice_tolerance: float = 1e-3
    target_encoding_m: float = 10.0

    export_preprocessed: bool = True

    def __post_init__(self):
        os.makedirs(self.results_dir, exist_ok=True)
        os.makedirs(self.preprocessed_dir, exist_ok=True)
        os.makedirs(self.missing_dir, exist_ok=True)
        os.makedirs(self.anova_dir, exist_ok=True)
        os.makedirs(self.tables_dir, exist_ok=True)
        os.makedirs(self.rankings_dir, exist_ok=True)
        os.makedirs(self.graphs_dir, exist_ok=True)

        self.rng = default_rng(self.seed)

CONFIG = ExperimentConfig()

## 3. MCAR Missing Data Generation

In [ ]:
class MCARGenerator:    
    def __init__(self, config: ExperimentConfig):
        self.config = config
        self.rng = config.rng
    
    def generate_missing_data(
        self, 
        df: pd.DataFrame, 
        features: List[str],
        nested: bool = True,
        filename_prefix: str = "heart_"
    ) -> Dict[str, pd.DataFrame]:
        X = df.copy(deep=True)
        
        numeric_features = [c for c in features if is_numeric_dtype(X[c])]
        n_rows, n_numeric_cols = len(X), len(numeric_features)
        total_numeric_cells = n_rows * n_numeric_cols
        
        if nested:
            all_cell_indices = self.rng.permutation(total_numeric_cells)
        
        outputs = {}
        
        for rate in self.config.mcar_rates:
            n_missing_cells = int(np.floor(rate * total_numeric_cells))
            
            if nested:
                chosen_cells = all_cell_indices[:n_missing_cells]
            else:
                chosen_cells = self.rng.choice(
                    total_numeric_cells, 
                    size=n_missing_cells, 
                    replace=False
                )
            
            rows = chosen_cells // n_numeric_cols
            cols = chosen_cells % n_numeric_cols
            
            mask = np.zeros((n_rows, n_numeric_cols), dtype=bool)
            mask[rows, cols] = True
            mask_df = pd.DataFrame(
                mask, 
                index=X.index, 
                columns=numeric_features
            )
            Y = X.copy(deep=True)
            Y[numeric_features] = Y[numeric_features].mask(mask_df, other=np.nan)
            
            pct = int(rate * 100)
            fname = f"{filename_prefix}{pct}.csv"
            fpath = os.path.join(self.config.missing_dir, fname)
            Y.to_csv(fpath, index=False)
            
            outputs[fname] = Y
        
        return outputs

## 4. Target Encoder Implementation

In [ ]:
class TargetEncoder:
    def __init__(self, cols: Optional[List[str]] = None, m: float = 10.0):
        self.cols = cols
        self.m = float(m)
        self.global_mean_ = None
        self.encoding_maps_ = {}
        self.fallback_values_ = {}
        self._is_fitted = False
    
    def fit(self, X, y):
        X = pd.DataFrame(X).copy()
        
        if self.cols is None:
            self.cols = X.select_dtypes(
                include=["object", "category", "string"]
            ).columns.tolist()
        
        y = pd.Series(y).astype(float)
        self.global_mean_ = y.mean()
        
        for col in self.cols:
            temp_df = pd.DataFrame({
                "category": X[col].astype("category"),
                "target": y
            })
            
            stats = temp_df.groupby("category", observed=True)["target"].agg([
                "sum", "count"
            ])
            
            smoothed_mean = (
                (stats["sum"] + self.m * self.global_mean_) / 
                (stats["count"] + self.m)
            )
            
            self.encoding_maps_[col] = smoothed_mean.to_dict()
            self.fallback_values_[col] = self.global_mean_
        
        self._is_fitted = True
        return self
    
    def transform(self, X):
       
        if not self._is_fitted:
            raise RuntimeError("The encoder has not been fitted. Call 'fit' before 'transform'.")
        
        X = pd.DataFrame(X).copy()
        
        for col in self.cols:
            X[col] = (X[col]
                     .map(self.encoding_maps_[col])
                     .fillna(self.fallback_values_[col])
                     .astype(float))
        
        return X
    
    def fit_transform(self, X, y):
        return self.fit(X, y).transform(X)

## 5. Preprocessing Pipeline Construction

In [ ]:
class PreprocessingFactory:
    def __init__(self, config: ExperimentConfig):
        self.config = config

    def get_encoder(self, encoding_type: str, cat_cols: List[str]):
        encoders = {
            "ordinal": OrdinalEncoder(
                handle_unknown="use_encoded_value", 
                unknown_value=-1
            ),
            "onehot": OneHotEncoder(
                handle_unknown="ignore", 
                sparse_output=False
            ),
            "target": TargetEncoder(
                cols=cat_cols, 
                m=self.config.target_encoding_m
            )
        }
        
        return ("encoder", encoders[encoding_type], cat_cols)
    
    def get_imputer(self, imputation_type: str):
        imputers = {
            "mean": SimpleImputer(strategy="mean"),
            "knn": KNNImputer(
                n_neighbors=self.config.knn_neighbors,
                weights="uniform"
            ),
            "mice": IterativeImputer(
                estimator=BayesianRidge(),
                max_iter=self.config.mice_max_iter,
                tol=self.config.mice_tolerance,
                initial_strategy="mean",
                imputation_order="ascending",
                random_state=self.config.seed,
                skip_complete=True
            )
        }
        
        return imputers[imputation_type]
    
    def get_scaler(self, scaling_type: str):
        scalers = {
            "zscore": StandardScaler(),
            "minmax": MinMaxScaler(),
            "robust": RobustScaler()
        }
        
        return scalers[scaling_type]
    
    def build_preprocessor(
        self, 
        encoding: str, 
        imputation: str, 
        scaling: str,
        df: pd.DataFrame
    ) -> Tuple[ColumnTransformer, List[str], List[str]]:
        num_cols, cat_cols = self._split_columns(df)
        
        numeric_pipeline = Pipeline([
            ("imputer", self.get_imputer(imputation)),
            ("scaler", self.get_scaler(scaling))
        ])
        
        _, encoder, _ = self.get_encoder(encoding, cat_cols)
        categorical_pipeline = Pipeline([("encoder", encoder)])
        
        preprocessor = ColumnTransformer(
            transformers=[
                ("num", numeric_pipeline, num_cols),
                ("cat", categorical_pipeline, cat_cols)
            ],
            remainder="drop",
            sparse_threshold=0.0,
            verbose_feature_names_out=False
        )
        
        return preprocessor, num_cols, cat_cols
    
    def _split_columns(self, df: pd.DataFrame) -> Tuple[List[str], List[str]]:
        features = [c for c in df.columns if c != self.config.target_column]
        
        num_cols = df[features].select_dtypes(
            include=[np.number, "float", "int", "Int64"]
        ).columns.tolist()
        
        cat_cols = [c for c in features if c not in num_cols]
        
        return num_cols, cat_cols
    
    def get_feature_names(
        self, 
        preprocessor: ColumnTransformer,
        num_cols: List[str],
        cat_cols: List[str]
    ) -> List[str]:
        names = num_cols.copy()
        
        if cat_cols:
            encoder = preprocessor.named_transformers_["cat"].named_steps["encoder"]
            
            if isinstance(encoder, OneHotEncoder):
                names += encoder.get_feature_names_out(cat_cols).tolist()
            else:
                names += cat_cols
        
        return names

## 6. ANOVA Analyzer

### 6.1. Class Definition

In [ ]:
class ANOVAAnalyzer:    
    def __init__(self, config: ExperimentConfig, fold_data: pd.DataFrame):
        self.config = config
        self.fold_data = self._prepare_data(fold_data)
    
    def _prepare_data(self, df: pd.DataFrame) -> pd.DataFrame:
        df_copy = df.copy()
        
        categorical_cols = [
            "encoding", "imputation", "scaler", 
            "model", "dataset", "mcar_pct"
        ]
        
        for col in categorical_cols:
            df_copy[col] = df_copy[col].astype("category")
        
        return df_copy
    
    def run_factorial_anova(
        self,
        metric: str = "mcc",
        per_mcar: bool = False
    ) -> List[Dict[str, Any]]:
       
        data = self.fold_data[self.fold_data["metric"] == metric].copy()       
        
        results = []
        
        if per_mcar:
            for mcar_pct, group_data in data.groupby("mcar_pct"):
                result = self._run_anova_block(group_data, label=f"mcar={mcar_pct}")
                results.append(result)
        else:
            result = self._run_anova_block(data, label="overall")
            results.append(result)
        
        return results

### 6.2. ANOVA Execution

In [ ]:
def _run_anova_block(self, data: pd.DataFrame, label: str = "overall") -> Dict[str, Any]:
        
        formula_parts = [
            "C(encoding)",
            "C(imputation)",
            "C(scaler)",
            "C(encoding):C(imputation)",
            "C(encoding):C(scaler)",
            "C(imputation):C(scaler)"
        ]
        
        formula = "value ~ " + " + ".join(formula_parts)
        
        model = smf.ols(formula, data=data).fit()
        anova_table = anova_lm(model, typ=2)
        
        self._save_anova_results(anova_table, label)
        
        return {
            "label": label,
            "anova": anova_table,
        }

ANOVAAnalyzer._run_anova_block = _run_anova_block

### 6.3. Exporting Results

In [ ]:
def _save_anova_results(self, anova_table: pd.DataFrame, label: str):
    clean_label = label.replace(" ", "").replace("=", "")
    anova_path = os.path.join(self.config.anova_dir, f"anova_{clean_label}.csv")
    anova_table.to_csv(anova_path)


ANOVAAnalyzer._save_anova_results = _save_anova_results

## 7. Experiment Evaluator

### 7.1. Initialization

In [ ]:
class ExperimentEvaluator:

    def __init__(self, config: ExperimentConfig):
        self.config = config
        self.factory = PreprocessingFactory(config)
        self.summary_results = []
        self.fold_results = []
        
        self.encodings = ["ordinal", "onehot", "target"]
        self.imputations = ["mean", "knn", "mice"]
        self.scalings = ["zscore", "minmax", "robust"]
        
        self.models = {
            "nb": GaussianNB(),
            "knn": KNeighborsClassifier(n_neighbors=config.knn_neighbors),
            "svm": SVC(
                probability=True, 
                kernel="rbf", 
                C=1.0, 
                random_state=config.seed
            )
        }
        
        self.scorers = {
            "f1": "f1",
            "mcc": "matthews_corrcoef"
        }
        
        self.cv = StratifiedKFold(
            n_splits=config.cv_folds,
            shuffle=True,
            random_state=config.seed
        )

### 7.2. Main Methods

In [ ]:
def run_experiment(self, input_files: List[str]):
        
        self._validate_input_files(input_files)
        
        for infile in input_files:
            self._process_dataset(infile)
        
        self._save_results()
        
        self._run_anova_analysis()

def _validate_input_files(self, input_files: List[str]):
        missing = [f for f in input_files if not os.path.exists(f)]
        if missing:
            raise FileNotFoundError(
                f"Files not found: {missing}"
            )

def _process_dataset(self, infile: str):   
        df = pd.read_csv(infile)
        if self.config.target_column not in df.columns:
            raise RuntimeError(
                f"'{self.config.target_column}' not found in {infile}"
            )
        
        mcar_pct = self._extract_mcar_percentage(infile)
      
        X = df.drop(columns=[self.config.target_column])
        y = df[self.config.target_column].values
        
        total_combinations = (
            len(self.encodings) * len(self.imputations) * 
            len(self.scalings) * len(self.models)
        )

        combination_count = 0
        for enc in self.encodings:
            for imp in self.imputations:
                for scl in self.scalings:
                    for mdl_name, mdl in self.models.items():
                        combination_count += 1
                        progress = f"[{combination_count}/{total_combinations}]"
                        
                        self._evaluate_combination(
                            X, y, df, infile, mcar_pct,
                            enc, imp, scl, mdl_name, mdl,
                            progress
                        )

def _extract_mcar_percentage(self, filename: str) -> Optional[int]:
    match = re.search(r"heart_(\d+)\.csv$", os.path.basename(filename))
    return int(match.group(1)) if match else None


def _run_anova_analysis(self):
    anova_analyzer = ANOVAAnalyzer(self.config, pd.DataFrame(self.fold_results))
    anova_analyzer.run_factorial_anova(metric="mcc", per_mcar=True)


ExperimentEvaluator.run_experiment = run_experiment
ExperimentEvaluator._validate_input_files = _validate_input_files
ExperimentEvaluator._process_dataset = _process_dataset
ExperimentEvaluator._extract_mcar_percentage = _extract_mcar_percentage
ExperimentEvaluator._run_anova_analysis = _run_anova_analysis

### 7.3. Evaluation

In [ ]:
def _evaluate_combination(
        self, X, y, df, infile, mcar_pct,
        enc, imp, scl, mdl_name, mdl, progress
    ):

        try:
            preproc, num_cols, cat_cols = self.factory.build_preprocessor(
                enc, imp, scl, df
            )
            pipeline = Pipeline([
                ("prep", preproc),
                ("clf", mdl)
            ])

            if self.config.export_preprocessed:
                scores = self._manual_cv_with_export(
                    pipeline, X, y, df, infile, enc, imp, scl, mdl_name,
                    num_cols, cat_cols
                )
            else:
                scores = cross_validate(
                    pipeline, X, y,
                    cv=self.cv,
                    scoring=self.scorers,
                    n_jobs=-1,
                    return_train_score=False
                )
            
            self._save_summary_results(
                infile, mcar_pct, enc, imp, scl, mdl_name,
                scores
            )
            
            self._save_fold_results(
                infile, mcar_pct, enc, imp, scl, mdl_name, scores
            )
            
            f1_mean = scores["test_f1"].mean()
            mcc_mean = scores["test_mcc"].mean()
            print(
                f"{progress} {enc}/{imp}/{scl}/{mdl_name} → "
                f"F1={f1_mean:.4f}, MCC={mcc_mean:.4f}"
            )
            
        except Exception as e:
            print(
                f"{progress} {enc}/{imp}/{scl}/{mdl_name} → "
                f"{type(e).__name__}: {e}"
            )

def _save_summary_results(
    self, infile, mcar_pct, enc, imp, scl, mdl_name,
    scores
):
        self.summary_results.append({
            "dataset": os.path.basename(infile),
            "mcar_pct": mcar_pct,
            "encoding": enc,
            "imputation": imp,
            "scaler": scl,
            "model": mdl_name,
            "f1_mean": scores["test_f1"].mean(),
            "f1_std": scores["test_f1"].std(),
            "mcc_mean": scores["test_mcc"].mean(),
            "mcc_std": scores["test_mcc"].std()
        })
    
def _save_fold_results(
        self, infile, mcar_pct, enc, imp, scl, mdl_name, scores
    ):
        n_folds = len(scores["test_f1"])
        
        for fold_idx in range(n_folds):
            self.fold_results.append({
                "dataset": os.path.basename(infile),
                "mcar_pct": mcar_pct,
                "encoding": enc,
                "imputation": imp,
                "scaler": scl,
                "model": mdl_name,
                "fold": fold_idx,
                "metric": "f1",
                "value": float(scores["test_f1"][fold_idx])
            })
            self.fold_results.append({
                "dataset": os.path.basename(infile),
                "mcar_pct": mcar_pct,
                "encoding": enc,
                "imputation": imp,
                "scaler": scl,
                "model": mdl_name,
                "fold": fold_idx,
                "metric": "mcc",
                "value": float(scores["test_mcc"][fold_idx])
            })

ExperimentEvaluator._evaluate_combination = _evaluate_combination
ExperimentEvaluator._save_summary_results = _save_summary_results
ExperimentEvaluator._save_fold_results = _save_fold_results

### 7.4. Exporting Results

In [ ]:
def _manual_cv_with_export(
    self, pipeline, X, y, df, infile, enc, imp, scl, 
    mdl_name, num_cols, cat_cols
):
    from sklearn.metrics import f1_score, matthews_corrcoef
    
    f1_scores = []
    mcc_scores = []
    
    base_name = os.path.splitext(os.path.basename(infile))[0]
    feature_names = None
    
    for fold_idx, (train_idx, test_idx) in enumerate(self.cv.split(X, y)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        X_train_transformed = pipeline.named_steps["prep"].fit_transform(X_train, y_train)
        X_test_transformed = pipeline.named_steps["prep"].transform(X_test)
        
        if feature_names is None:
            feature_names = self.factory.get_feature_names(
                pipeline.named_steps["prep"], num_cols, cat_cols
            )
        
        X_train_transformed = np.asarray(X_train_transformed)
        X_test_transformed = np.asarray(X_test_transformed)
        
        train_df = pd.DataFrame(
            X_train_transformed,
            columns=feature_names,
            index=X_train.index
        )
        train_df.insert(0, self.config.target_column, y_train)
        
        test_df = pd.DataFrame(
            X_test_transformed,
            columns=feature_names,
            index=X_test.index
        )
        test_df.insert(0, self.config.target_column, y_test)
        
        train_path = os.path.join(
            self.config.preprocessed_dir,
            f"{base_name}_{enc}_{imp}_{scl}_{mdl_name}_fold{fold_idx}_train.csv"
        )
        test_path = os.path.join(
            self.config.preprocessed_dir,
            f"{base_name}_{enc}_{imp}_{scl}_{mdl_name}_fold{fold_idx}_test.csv"
        )
        
        train_df.to_csv(train_path, index=False)
        test_df.to_csv(test_path, index=False)
        
        pipeline.named_steps["clf"].fit(X_train_transformed, y_train)
        y_pred = pipeline.named_steps["clf"].predict(X_test_transformed)
        
        f1_scores.append(f1_score(y_test, y_pred))
        mcc_scores.append(matthews_corrcoef(y_test, y_pred))
    
    return {
        "test_f1": np.array(f1_scores),
        "test_mcc": np.array(mcc_scores)
    }

def _save_results(self):
    summary_df = pd.DataFrame(self.summary_results)
    summary_df_sorted = summary_df.sort_values(
            by=["mcar_pct", "model", "mcc_mean", "f1_mean"],
            ascending=[True, True, False, False]
        )

    for pct, group in summary_df_sorted.groupby("mcar_pct"):
        filename = os.path.join(
                self.config.results_dir,
                f"results_heart_{pct}.csv"
            )
        group.to_csv(filename, index=False)

    all_results_path = os.path.join(
            self.config.results_dir,
            "results_all.csv"
        )
    summary_df_sorted.to_csv(all_results_path, index=False)

    fold_df = pd.DataFrame(self.fold_results)
    fold_results_path = os.path.join(
            self.config.results_dir,
            "fold_scores_all.csv"
        )
    fold_df.to_csv(fold_results_path, index=False)

ExperimentEvaluator._manual_cv_with_export = _manual_cv_with_export
ExperimentEvaluator._save_results = _save_results

## 8. Results Analysis & Visualization

### 8.1. Initialization

In [ ]:
plt.style.use("seaborn-v0_8-paper")
sns.set_palette("husl")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 10

class ResultsAnalyzer:
    def __init__(self, results_dir: str = "results", graphs_subdir: str = "graphs"):
        self.results_dir = results_dir
        self.anova_dir = os.path.join(results_dir, "anova")
        self.tables_dir = os.path.join(results_dir, "tables")
        self.rankings_dir = os.path.join(results_dir, "tables", "rankings")
        self.graphs_dir = os.path.join(results_dir, graphs_subdir)
        self.results_all = None
        self.fold_scores = None

        self.load_results()

    def load_results(self):
        try:
            self.results_all = pd.read_csv(f"{self.results_dir}/results_all.csv")
            self.fold_scores = pd.read_csv(f"{self.results_dir}/fold_scores_all.csv")
            print(f"Results loaded: {len(self.results_all)} combinations")
        except FileNotFoundError:
            print(
                "No results files found."
            )

### 8.2. Significant Effects Detection

In [ ]:
def get_significant_effects(self, alpha: float = 0.05) -> Dict[str, Dict]:

    factor_mapping = {
        "C(encoding)": "encoding",
        "C(imputation)": "imputation",
        "C(scaler)": "scaler",
        "C(encoding):C(imputation)": "encoding:imputation",
        "C(encoding):C(scaler)": "encoding:scaler",
        "C(imputation):C(scaler)": "imputation:scaler",
    }

    results = {}

    for mcar_pct in [10, 20, 30]:
        anova_file = f"anova_mcar{mcar_pct}.csv"
        anova_path = os.path.join(self.anova_dir, anova_file)

        try:
            anova_df = pd.read_csv(anova_path, index_col=0)
        except FileNotFoundError:
            print(f"ANOVA file not found: {anova_path}")
            continue

        main_effects = []
        interactions = []
        details = {}

        for anova_key, readable_name in factor_mapping.items():
            if anova_key in anova_df.index:
                f_val = anova_df.loc[anova_key, "F"]
                p_val = anova_df.loc[anova_key, "PR(>F)"]

                details[readable_name] = {
                    "F": f_val,
                    "p": p_val,
                    "significant": p_val < alpha if pd.notna(p_val) else False,
                }

                if pd.notna(p_val) and p_val < alpha:
                    if ":" in readable_name:
                        interactions.append(readable_name)
                    else:
                        main_effects.append(readable_name)

        results[mcar_pct] = {
            "main_effects": main_effects,
            "interactions": interactions,
            "details": details,
        }

    return results

ResultsAnalyzer.get_significant_effects = get_significant_effects

### 8.3. Consolidated ANOVA Table

In [ ]:
def generate_anova_consolidated_table(self, alpha: float = 0.05, save: bool = True):

    factor_display = {
        "encoding": "Encoding",
        "imputation": "Imputation",
        "scaler": "Scaling",
        "encoding:imputation": "Encoding × Imputation",
        "encoding:scaler": "Encoding × Scaling",
        "imputation:scaler": "Imputation × Scaling",
    }

    significant_effects = self.get_significant_effects(alpha)

    rows = []

    rows.append(
        {
            "Factor / Interaction": "Main effects",
            "10% MCAR": "",
            "20% MCAR": "",
            "30% MCAR": "",
        }
    )

    for factor in ["encoding", "scaler", "imputation"]:
        row = {"Factor / Interaction": factor_display[factor]}
        for mcar_pct in [10, 20, 30]:
            if mcar_pct in significant_effects:
                details = significant_effects[mcar_pct]["details"]
                if factor in details:
                    f_val = details[factor]["F"]
                    p_val = details[factor]["p"]
                    sig = (
                        "***"
                        if p_val < 0.001
                        else "**" if p_val < 0.01 else "*" if p_val < 0.05 else ""
                    )
                    row[f"{mcar_pct}% MCAR"] = (
                        f"F={f_val:.2f}, p={p_val:.4f}{sig}" if pd.notna(f_val) else "N/A"
                    )
                else:
                    row[f"{mcar_pct}% MCAR"] = "N/A"
            else:
                row[f"{mcar_pct}% MCAR"] = "N/A"
        rows.append(row)

    rows.append(
        {
            "Factor / Interaction": "Interactions",
            "10% MCAR": "",
            "20% MCAR": "",
            "30% MCAR": "",
        }
    )

    for interaction in ["encoding:scaler", "encoding:imputation", "imputation:scaler"]:
        row = {"Factor / Interaction": factor_display[interaction]}
        for mcar_pct in [10, 20, 30]:
            if mcar_pct in significant_effects:
                details = significant_effects[mcar_pct]["details"]
                if interaction in details:
                    f_val = details[interaction]["F"]
                    p_val = details[interaction]["p"]
                    sig = (
                        "***"
                        if p_val < 0.001
                        else "**" if p_val < 0.01 else "*" if p_val < 0.05 else ""
                    )
                    row[f"{mcar_pct}% MCAR"] = (
                        f"F={f_val:.2f}, p={p_val:.4f}{sig}" if pd.notna(f_val) else "N/A"
                    )
                else:
                    row[f"{mcar_pct}% MCAR"] = "N/A"
            else:
                row[f"{mcar_pct}% MCAR"] = "N/A"
        rows.append(row)

    result_df = pd.DataFrame(rows)

    if save:
        result_df.to_csv(
            os.path.join(self.tables_dir, "table_anova_consolidated.csv"), index=False
        )

    return result_df, significant_effects

ResultsAnalyzer.generate_anova_consolidated_table = generate_anova_consolidated_table

### 8.4. Main Effect and Interaction Tables

In [ ]:
def generate_main_effect_table(self, factor: str, save: bool = True):
    
    factor_display = {
        "encoding": "Encoder",
        "imputation": "Imputation",
        "scaler": "Scaler",
    }

    level_display = {
        "encoding": {"onehot": "One-Hot", "ordinal": "Ordinal", "target": "Target"},
        "imputation": {"mean": "Mean", "knn": "kNN", "mice": "MICE"},
        "scaler": {"zscore": "Z-Score", "minmax": "Min-Max", "robust": "Robust"},
    }

    rows = []
    levels = level_display[factor].keys()

    for level in levels:
        for metric, metric_col in [("F1", "f1_mean"), ("MCC", "mcc_mean")]:
            row = {
                factor_display[factor]: level_display[factor][level],
                "Metric": metric,
            }

            for mcar_pct in [10, 20, 30]:
                subset = self.results_all[
                    (self.results_all[factor] == level)
                    & (self.results_all["mcar_pct"] == mcar_pct)
                ]

                mean_val = subset[metric_col].mean()
                std_val = subset[metric_col].std()
                row[f"{mcar_pct}% MCAR"] = f"{mean_val:.3f} ± {std_val:.3f}"

            rows.append(row)

    result_df = pd.DataFrame(rows)

    if save:
        result_df.to_csv(
            os.path.join(self.tables_dir, f"table_effect_{factor}.csv"), index=False
        )

    return result_df


def generate_interaction_table(self, interaction: str, significant_mcar_levels: List[int] = None, save: bool = True):
    
    if significant_mcar_levels is None or len(significant_mcar_levels) == 0:
        return None
    
    factor_a, factor_b = interaction.split(":")

    level_display = {
        "encoding": {"onehot": "One-Hot", "ordinal": "Ordinal", "target": "Target"},
        "imputation": {"mean": "Mean", "knn": "kNN", "mice": "MICE"},
        "scaler": {"zscore": "Z-Score", "minmax": "Min-Max", "robust": "Robust"},
    }

    factor_display = {
        "encoding": "Encoder",
        "imputation": "Imputation",
        "scaler": "Scaler",
    }

    rows = []

    for mcar_pct in significant_mcar_levels:
        for level_a in level_display[factor_a].keys():
            for metric, metric_col in [("F1", "f1_mean"), ("MCC", "mcc_mean")]:
                row = {
                    factor_display[factor_a]: level_display[factor_a][level_a],
                    "Metric": metric
                }

                for level_b in level_display[factor_b].keys():
                    subset = self.results_all[
                        (self.results_all[factor_a] == level_a)
                        & (self.results_all[factor_b] == level_b)
                        & (self.results_all["mcar_pct"] == mcar_pct)
                    ]

                    mean_val = subset[metric_col].mean()
                    std_val = subset[metric_col].std()

                    col_name = level_display[factor_b][level_b]
                    row[col_name] = f"{mean_val:.3f} ± {std_val:.3f}"

                row["MCAR %"] = mcar_pct

                rows.append(row)

    result_df = pd.DataFrame(rows)

    if save:
        interaction_name = interaction.replace(":", "_x_")
        result_df.to_csv(
            os.path.join(
                self.tables_dir, f"table_interaction_{interaction_name}.csv"
            ),
            index=False,
        )

    return result_df


ResultsAnalyzer.generate_main_effect_table = generate_main_effect_table
ResultsAnalyzer.generate_interaction_table = generate_interaction_table

### 8.5. Generating Significant Effect Tables

In [ ]:
def generate_significant_effect_tables(self, alpha: float = 0.05, save: bool = True):
    
    significant_effects = self.get_significant_effects(alpha)

    all_main_effects = set()
    all_interactions = set()

    for mcar_pct, effects in significant_effects.items():
        all_main_effects.update(effects["main_effects"])
        all_interactions.update(effects["interactions"])

    generated_tables = {"main_effects": {}, "interactions": {}}

    for factor in all_main_effects:
        table = self.generate_main_effect_table(factor, save)
        generated_tables["main_effects"][factor] = table

    for interaction in all_interactions:
        significant_mcar_levels = []
        for mcar_pct, effects in significant_effects.items():
            if interaction in effects["interactions"]:
                significant_mcar_levels.append(mcar_pct)
        
        table = self.generate_interaction_table(interaction, significant_mcar_levels, save)
        if table is not None:
            generated_tables["interactions"][interaction] = table

    return generated_tables, significant_effects

ResultsAnalyzer.generate_significant_effect_tables = generate_significant_effect_tables

### 8.6. Combination Ranking Tables

In [ ]:
def generate_ranked_combinations_table(self, mcar_pct: int = 10, save: bool = True):

    mcar_data = self.results_all[self.results_all["mcar_pct"] == mcar_pct].copy()

    encoding_map = {"onehot": "One-Hot", "ordinal": "Ordinal", "target": "Target"}
    imputation_map = {"mean": "Mean", "knn": "kNN", "mice": "MICE"}
    scaler_map = {"zscore": "Z-Score", "minmax": "Min-Max", "robust": "Robust"}
    model_map = {"nb": "NB", "knn": "kNN", "svm": "SVM"}

    sorted_data = mcar_data.sort_values("mcc_mean", ascending=False)

    ranked_table = sorted_data[["encoding", "imputation", "scaler", "model", "mcc_mean", "f1_mean"]].copy()
    ranked_table["encoding"] = ranked_table["encoding"].map(encoding_map)
    ranked_table["imputation"] = ranked_table["imputation"].map(imputation_map)
    ranked_table["scaler"] = ranked_table["scaler"].map(scaler_map)
    ranked_table["model"] = ranked_table["model"].map(model_map)
    ranked_table.columns = ["Encoding", "Imputation", "Scaler", "Model", "MCC", "F1"]
    ranked_table.insert(0, "Rank", range(1, len(ranked_table) + 1))
    ranked_table["MCC"] = ranked_table["MCC"].apply(lambda x: f"{x:.3f}")
    ranked_table["F1"] = ranked_table["F1"].apply(lambda x: f"{x:.3f}")
    ranked_table = ranked_table.reset_index(drop=True)

    if save:
        ranked_table.to_csv(
            os.path.join(self.rankings_dir, f"table_ranked_all_mcar{mcar_pct}.csv"),
            index=False,
        )

    return ranked_table


def generate_all_ranked_tables(self, save: bool = True):
    results = {}
    for mcar_pct in [10, 20, 30]:
        ranked = self.generate_ranked_combinations_table(mcar_pct, save)
        results[mcar_pct] = ranked
    return results


ResultsAnalyzer.generate_ranked_combinations_table = generate_ranked_combinations_table
ResultsAnalyzer.generate_all_ranked_tables = generate_all_ranked_tables

### 8.7. MCC Heatmaps

In [ ]:
def plot_mcc_heatmap_by_mcar(self, mcar_pct=10, save=True):
    fig, ax = plt.subplots(figsize=(12, 7))
    
    df_mcar = self.results_all[self.results_all["mcar_pct"] == mcar_pct]
    
    pivot = df_mcar.pivot_table(
        values="mcc_mean",
        index=["encoding", "imputation"],
        columns=["scaler", "model"],
        aggfunc="mean",
    )
    
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".3f",
        cmap="RdYlGn",
        vmin=0.35,
        vmax=0.70,
        ax=ax,
        cbar_kws={"label": "MCC"},
        annot_kws={"fontsize": 8},
    )
    
    ax.set_title(
        f"Matthews Correlation Coefficient (MCC) - MCAR {mcar_pct}%",
        fontsize=14,
        fontweight="bold",
        pad=15
    )
    ax.set_xlabel("Scaling / Model", fontsize=11)
    ax.set_ylabel("Encoding / Imputation", fontsize=11)
    ax.tick_params(axis="both", labelsize=9)
    
    plt.tight_layout()
    
    if save:
        filename = f"heatmap_mcc_mcar{mcar_pct}.png"
        plt.savefig(
            os.path.join(self.graphs_dir, filename),
            bbox_inches="tight",
            dpi=300
        )    
    plt.show()
    plt.close()


def plot_all_mcc_heatmaps(self, save=True):
    for mcar_pct in [10, 20, 30]:
        self.plot_mcc_heatmap_by_mcar(mcar_pct=mcar_pct, save=save)

ResultsAnalyzer.plot_mcc_heatmap_by_mcar = plot_mcc_heatmap_by_mcar
ResultsAnalyzer.plot_all_mcc_heatmaps = plot_all_mcc_heatmaps

### 8.8. F1-Score Heatmaps

In [ ]:
def plot_f1_heatmap_by_mcar(self, mcar_pct=10, save=True):
    fig, ax = plt.subplots(figsize=(12, 7))
    
    df_mcar = self.results_all[self.results_all["mcar_pct"] == mcar_pct]
    
    pivot = df_mcar.pivot_table(
        values="f1_mean",
        index=["encoding", "imputation"],
        columns=["scaler", "model"],
        aggfunc="mean",
    )
    
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".3f",
        cmap="RdYlGn",
        vmin=0.50,
        vmax=0.75,
        ax=ax,
        cbar_kws={"label": "F1-Score"},
        annot_kws={"fontsize": 8},
    )
    
    ax.set_title(
        f"F1-Score - MCAR {mcar_pct}%",
        fontsize=14,
        fontweight="bold",
        pad=15
    )
    ax.set_xlabel("Scaling / Model", fontsize=11)
    ax.set_ylabel("Encoding / Imputation", fontsize=11)
    ax.tick_params(axis="both", labelsize=9)
    
    plt.tight_layout()
    
    if save:
        filename = f"heatmap_f1_mcar{mcar_pct}.png"
        plt.savefig(
            os.path.join(self.graphs_dir, filename),
            bbox_inches="tight",
            dpi=300
        )    
    plt.show()
    plt.close()


def plot_all_f1_heatmaps(self, save=True):
    for mcar_pct in [10, 20, 30]:
        self.plot_f1_heatmap_by_mcar(mcar_pct=mcar_pct, save=save)

ResultsAnalyzer.plot_f1_heatmap_by_mcar = plot_f1_heatmap_by_mcar
ResultsAnalyzer.plot_all_f1_heatmaps = plot_all_f1_heatmaps

## 9. Main Function Definition & Execution

In [ ]:
def main():
    print("\n")
    print("╔" + "═" * 78 + "╗")
    print("║" + " " * 78 + "║")
    print("║" + "EFFECTS OF PREPROCESSING SCHEMES ON CLASSIFIER PERFORMANCE".center(78) + "║")
    print("║" + "UNDER VARYING PERCENTAGES OF MISSING VALUES".center(78) + "║")
    print("║" + " " * 78 + "║")
    print("║" + "Christian R. Carpio & Luis E. Benavides".center(78) + "║")
    print("║" + " " * 78 + "║")
    print("╚" + "═" * 78 + "╝")
    print()

    config = ExperimentConfig(export_preprocessed=True)

    print("EXPERIMENT CONFIGURATION")
    print("─" * 40)
    print(f"   • Random seed: {config.seed}")
    print(f"   • Target variable: {config.target_column}")
    print(f"   • MCAR rates: {[int(r*100) for r in config.mcar_rates]}%")
    print(f"   • Cross-validation folds: {config.cv_folds}")
    print(f"   • Results directory: {config.results_dir}/")
    print()

    print("LOADING ORIGINAL DATASET")
    print("─" * 40)
    df_original = pd.read_csv('heart.csv')

    print(f"   • Shape: {df_original.shape}")
    print(f"   • Columns: {list(df_original.columns)}")
    print()

    print("GENERATING MCAR MISSING-VALUE DATASETS")
    print("─" * 40)
    generator = MCARGenerator(config)
    generator.generate_missing_data(
        df=df_original,
        features=[c for c in df_original.columns if c != config.target_column],
        nested=True,
        filename_prefix="heart_"
    )
    print()

    input_files = [
        os.path.join(config.missing_dir, f"heart_{pct}.csv")
        for pct in (10, 20, 30)
    ]

    print("METHODS UNDER EVALUATION")
    print("─" * 40)
    print("   • Encoding: ordinal, onehot, target")
    print("   • Imputation: mean, knn, mice")
    print("   • Scaling: zscore, minmax, robust")
    print("   • Models: Naive Bayes, KNN, SVM")
    print(f"   • Total combinations: {3*3*3*3*len(input_files)} evaluations")
    print()

    print("RUNNING EXPERIMENT")
    print("─" * 40)
    evaluator = ExperimentEvaluator(config)
    evaluator.run_experiment(input_files)

    analyzer = ResultsAnalyzer()

    print("\n")
    print("GENERATING RESULT TABLES AND CHARTS")
    print("─" * 40)

    print("CONSOLIDATED FACTORIAL ANOVA")
    print("─" * 40)
    anova_table, significant_effects = analyzer.generate_anova_consolidated_table()
    print(anova_table.to_string(index=False))

    print("\n")
    print("SIGNIFICANT EFFECTS")
    print("─" * 40)
    generated_tables, _ = analyzer.generate_significant_effect_tables(save=True)
    print(f"   Generated {len(generated_tables['main_effects'])} main effect tables")
    print(f"   Generated {len(generated_tables['interactions'])} interaction tables")

    print("\n")
    print("MCC AND F1-SCORE HEATMAPS")
    print("─" * 40)

    analyzer.plot_all_mcc_heatmaps()

    analyzer.plot_all_f1_heatmaps()

    print("\n")
    print("COMBINATION RANKING BY MCC")
    print("─" * 40)
    rankings_results = analyzer.generate_all_ranked_tables(save=True)
    for mcar_pct in [10, 20, 30]:
        print(f"\n   Generated table for MCAR {mcar_pct}%")
        print(f"      Top 5 combinations:")
        print(rankings_results[mcar_pct].head(5).to_string(index=False))
        print(f"\n      Bottom 5 combinations:")
        print(rankings_results[mcar_pct].tail(5).to_string(index=False))

    return analyzer

In [ ]:
if __name__ == "__main__":
    evaluator = main()